In [1]:
print(0)

0


In [2]:
#!/usr/bin/env python3
"""
Dataset statistics:
  - Per-source image counts
  - Resolution stats: min / median / max for W, H, megapixels
  - Small-image counts (below 3 thresholds)
  - Images-per-case distribution (up to 6+)
"""

import os
import sys
import math
import statistics
from pathlib import Path
from collections import defaultdict

from PIL import Image

DATASET_ROOT = "/home/pratyushprasad/ACL/DATA/FINAL_DATASET"
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}

SMALL_THRESHOLDS = [
    (64,  64),
    (128, 128),
    (256, 256),
]

DATASETS = [
    "1.DERM_1M",
    "2.DERM_VQA",
    "3.MM_SKIN_QA",
    "4.PAD_UFES_20",
    "5.SCIN",
    "6.BIOMEDICA_PUBMED",
    "7.FITZPATRIK_17K",
    "8.MUHOMMAD_DATA",
    "9.YUHOS16_SKINCARE",
]


def find_images(root):
    """Recursively find all image paths under root."""
    paths = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if Path(f).suffix.lower() in IMAGE_EXTS:
                paths.append(os.path.join(dirpath, f))
    return sorted(paths)


def read_dims(path):
    """Return (width, height) or None on failure."""
    try:
        with Image.open(path) as im:
            return im.size  # (width, height)
    except Exception:
        return None


def percentile(data, p):
    """Simple percentile (0-100) on a sorted list."""
    if not data:
        return float("nan")
    s = sorted(data)
    idx = (len(s) - 1) * p / 100
    lo, hi = int(idx), math.ceil(idx)
    if lo == hi:
        return s[lo]
    return s[lo] * (hi - idx) + s[hi] * (idx - lo)


def fmt(v):
    if isinstance(v, float) and math.isnan(v):
        return "  N/A  "
    return f"{v:>7.0f}"


def fmt_mp(v):
    if isinstance(v, float) and math.isnan(v):
        return "  N/A  "
    return f"{v:>7.3f}"


def images_per_case(paths):
    """
    Count images per 'case' = immediate parent directory.
    Returns a dict {bucket: count} where bucket is 1..6 and '7+'.
    """
    case_counts = defaultdict(int)
    for p in paths:
        parent = os.path.dirname(p)
        case_counts[parent] += 1

    distribution = defaultdict(int)
    for n in case_counts.values():
        bucket = str(min(n, 6)) if n <= 6 else "7+"
        distribution[bucket] += 1
    return distribution


# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("\n" + "=" * 90)
    print("  DATASET STATISTICS")
    print("=" * 90)

    grand_widths, grand_heights, grand_mp = [], [], []
    grand_total = 0
    grand_small = [0] * len(SMALL_THRESHOLDS)

    summary_rows = []

    for dname in DATASETS:
        dpath = os.path.join(DATASET_ROOT, dname)
        if not os.path.isdir(dpath):
            print(f"\n  [WARN] Not found: {dpath}")
            continue

        paths = find_images(dpath)
        n_total = len(paths)
        print(f"\n  {dname}  ({n_total} images) — reading dims...", flush=True)

        widths, heights, mps = [], [], []
        errors = 0

        for i, p in enumerate(paths, 1):
            dims = read_dims(p)
            if dims is None:
                errors += 1
                continue
            w, h = dims
            widths.append(w)
            heights.append(h)
            mps.append(w * h / 1_000_000)
            if (i % 5000) == 0:
                print(f"    ... {i}/{n_total}", flush=True)

        grand_widths  += widths
        grand_heights += heights
        grand_mp      += mps
        grand_total   += n_total

        # ── small image counts ───────────────────────────────────────────────
        small_counts = []
        for tw, th in SMALL_THRESHOLDS:
            c = sum(1 for w, h in zip(widths, heights) if w < tw or h < th)
            small_counts.append(c)
            grand_small[SMALL_THRESHOLDS.index((tw, th))] += c

        # ── images per case ──────────────────────────────────────────────────
        ipc = images_per_case(paths)

        # ── print per-dataset block ──────────────────────────────────────────
        print(f"\n  ── {dname} ──")
        print(f"     Total images  : {n_total}  (read errors: {errors})")
        print(f"     {'':30s} {'Min':>8} {'Median':>8} {'Max':>8}")
        print(f"     {'Width  (px)':30s} {fmt(min(widths) if widths else float('nan'))} "
              f"{fmt(percentile(widths,50))} {fmt(max(widths) if widths else float('nan'))}")
        print(f"     {'Height (px)':30s} {fmt(min(heights) if heights else float('nan'))} "
              f"{fmt(percentile(heights,50))} {fmt(max(heights) if heights else float('nan'))}")
        print(f"     {'Megapixels':30s} {fmt_mp(min(mps) if mps else float('nan'))} "
              f"{fmt_mp(percentile(mps,50))} {fmt_mp(max(mps) if mps else float('nan'))}")
        print(f"     Small images:")
        for (tw, th), sc in zip(SMALL_THRESHOLDS, small_counts):
            pct = 100 * sc / n_total if n_total else 0
            print(f"       < {tw}×{th:>3}px  :  {sc:>6}  ({pct:.1f}%)")

        print(f"     Images per case (parent dir):")
        for bucket in ["1","2","3","4","5","6","7+"]:
            cnt = ipc.get(bucket, 0)
            if cnt > 0:
                print(f"       {bucket:>3} image(s)  :  {cnt} cases")

        summary_rows.append({
            "dataset": dname,
            "n": n_total,
            "w_min": min(widths) if widths else float("nan"),
            "w_med": percentile(widths, 50),
            "w_max": max(widths) if widths else float("nan"),
            "h_min": min(heights) if heights else float("nan"),
            "h_med": percentile(heights, 50),
            "h_max": max(heights) if heights else float("nan"),
            "mp_min": min(mps) if mps else float("nan"),
            "mp_med": percentile(mps, 50),
            "mp_max": max(mps) if mps else float("nan"),
            "small_counts": small_counts,
        })

    # ── grand summary ─────────────────────────────────────────────────────────
    print("\n" + "=" * 90)
    print("  GRAND SUMMARY  (all datasets combined)")
    print("=" * 90)
    print(f"  Total images  : {grand_total}")
    print(f"  {'':30s} {'Min':>8} {'Median':>8} {'Max':>8}")
    print(f"  {'Width  (px)':30s} {fmt(min(grand_widths) if grand_widths else float('nan'))} "
          f"{fmt(percentile(grand_widths,50))} {fmt(max(grand_widths) if grand_widths else float('nan'))}")
    print(f"  {'Height (px)':30s} {fmt(min(grand_heights) if grand_heights else float('nan'))} "
          f"{fmt(percentile(grand_heights,50))} {fmt(max(grand_heights) if grand_heights else float('nan'))}")
    print(f"  {'Megapixels':30s} {fmt_mp(min(grand_mp) if grand_mp else float('nan'))} "
          f"{fmt_mp(percentile(grand_mp,50))} {fmt_mp(max(grand_mp) if grand_mp else float('nan'))}")
    print(f"  Small images:")
    for (tw, th), sc in zip(SMALL_THRESHOLDS, grand_small):
        pct = 100 * sc / grand_total if grand_total else 0
        print(f"    < {tw}×{th:>3}px  :  {sc:>7}  ({pct:.1f}%)")

    print("\n  Source composition:")
    for row in summary_rows:
        pct = 100 * row["n"] / grand_total if grand_total else 0
        print(f"    {row['dataset']:<35} {row['n']:>7} images  ({pct:.1f}%)")

    print("\n" + "=" * 90 + "\n")


if __name__ == "__main__":
    main()


  DATASET STATISTICS

  1.DERM_1M  (423016 images) — reading dims...
    ... 5000/423016
    ... 10000/423016
    ... 15000/423016
    ... 20000/423016
    ... 25000/423016
    ... 30000/423016
    ... 35000/423016
    ... 40000/423016
    ... 45000/423016
    ... 50000/423016
    ... 55000/423016
    ... 60000/423016
    ... 65000/423016
    ... 70000/423016
    ... 75000/423016
    ... 80000/423016
    ... 85000/423016
    ... 90000/423016
    ... 95000/423016
    ... 100000/423016
    ... 105000/423016
    ... 110000/423016
    ... 115000/423016
    ... 120000/423016
    ... 125000/423016
    ... 130000/423016
    ... 135000/423016
    ... 140000/423016
    ... 145000/423016
    ... 150000/423016
    ... 155000/423016
    ... 160000/423016
    ... 165000/423016
    ... 170000/423016
    ... 175000/423016
    ... 180000/423016
    ... 185000/423016
    ... 190000/423016
    ... 195000/423016
    ... 200000/423016
    ... 205000/423016
    ... 210000/423016
    ... 215000/423016
    